# GeoMaster on Google Colab
公開リポジトリを取得し、配信先を明示したHTTPサーバーでGeoMasterを起動します。GitHubトークンは不要です。

In [ ]:
from pathlib import Path
import os, shutil, subprocess

repo_dir = Path('/content/GeoMaster')

# 前回の実行で作業場所がGeoMaster内でも、削除前に安全な場所へ退避する。
os.chdir('/content')
if repo_dir.exists():
    shutil.rmtree(repo_dir)

result = subprocess.run([
    'git', 'clone', '--depth', '1',
    'https://github.com/TomTomYoung/GeoMaster.git',
    str(repo_dir)
], text=True, capture_output=True)

print(result.stdout, end='')
if result.returncode != 0:
    print(result.stderr, end='')
    raise RuntimeError(f'git clone failed: exit {result.returncode}')

index_file = repo_dir / 'index.html'
assert index_file.is_file(), f'index.html がありません: {index_file}'
print('Repository:', repo_dir)
print('Entry file:', index_file)


In [ ]:
import subprocess, time, urllib.request

PORT = 8000
old_server = globals().get('geomaster_server')
if old_server is not None and old_server.poll() is None:
    old_server.terminate()
    old_server.wait(timeout=5)

geomaster_server = subprocess.Popen(
    ['python', '-m', 'http.server', str(PORT), '--directory', str(repo_dir)],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True,
)

for _ in range(30):
    if geomaster_server.poll() is not None:
        raise RuntimeError('HTTPサーバーが終了しました:\n' + geomaster_server.stdout.read())
    try:
        with urllib.request.urlopen(f'http://127.0.0.1:{PORT}/', timeout=1) as response:
            body = response.read(200).decode('utf-8', errors='replace')
            print('HTTP status:', response.status)
            print('Response head:', body[:80])
            break
    except Exception:
        time.sleep(0.2)
else:
    geomaster_server.terminate()
    raise RuntimeError('HTTPサーバーへ接続できませんでした')


In [ ]:
from google.colab import output
output.serve_kernel_port_as_iframe(PORT, height=720)
